
# DLOps Assignment-1 (Pinned Memory Version)

**Student Name:** SHIVAM MADHAV KENCHE

**Roll Number:** M25CSA028

**Deadline:** January 24, 2026, 11:59 PM

---

## Assignment Overview
This notebook contains experiments with **pin_memory=True** for:
- **Q1(a):** Training ResNet-18 and ResNet-50 on MNIST and FashionMNIST
- **Q1(b):** Training SVM classifiers on MNIST and FashionMNIST
- **Q2:** CPU vs GPU performance comparison on FashionMNIST

**Note:** This version uses pinned memory for faster GPU data transfer.

## 1. Setup and Installation

## ⚠️ Important: Pinned Memory Configuration

This notebook version uses **`pin_memory=True`** for all data loaders to enable faster GPU data transfer.

### What is Pinned Memory?
- Pinned (page-locked) memory allows direct memory access (DMA) from CPU to GPU
- Provides faster data transfer compared to pageable memory
- Particularly beneficial for GPU training workloads

### Configuration Changes:
- All Q1(a) experiments use `pin_memory=True`
- Model names include `_pinned` suffix to distinguish from non-pinned versions
- Expected benefits: Reduced data loading overhead and faster training times

### Trade-offs:
- **Pros:** Faster GPU data transfer, reduced training time
- **Cons:** Higher host memory usage, may cause issues on systems with limited RAM

In [ ]:
# Install required packages (run in Colab)
# !pip install torch torchvision tqdm thop scikit-learn matplotlib seaborn pandas

In [1]:
# Import libraries
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

Using device: cuda
GPU: Tesla T4
CUDA Version: 12.6


In [2]:
"""
ResNet implementations for MNIST and FashionMNIST
Models: ResNet-18, ResNet-32, ResNet-50
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    """Basic residual block for ResNet-18/32"""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    """Bottleneck residual block for ResNet-50"""
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion,
                               kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    """ResNet architecture adapted for MNIST/FashionMNIST (28x28 images)"""

    def __init__(self, block, num_blocks, num_classes=10, in_channels=1):
        super(ResNet, self).__init__()
        self.in_channels = 64

        # Initial convolution - adapted for 28x28 images
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        # Residual layers
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        # Global average pooling and fully connected layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


def ResNet18(num_classes=10, in_channels=1, pretrained=False):
    """ResNet-18 for MNIST/FashionMNIST"""
    if pretrained:
        raise ValueError("Pretrained weights not supported for this assignment")
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes, in_channels=in_channels)


def ResNet32(num_classes=10, in_channels=1, pretrained=False):
    """ResNet-32 for MNIST/FashionMNIST"""
    if pretrained:
        raise ValueError("Pretrained weights not supported for this assignment")
    return ResNet(BasicBlock, [5, 5, 5, 5], num_classes=num_classes, in_channels=in_channels)


def ResNet50(num_classes=10, in_channels=1, pretrained=False):
    """ResNet-50 for MNIST/FashionMNIST"""
    if pretrained:
        raise ValueError("Pretrained weights not supported for this assignment")
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes, in_channels=in_channels)


# Test the models
if __name__ == "__main__":
    # Test ResNet-18
    model18 = ResNet18()
    x = torch.randn(1, 1, 28, 28)
    y = model18(x)
    print(f"ResNet-18 output shape: {y.shape}")
    print(f"ResNet-18 parameters: {sum(p.numel() for p in model18.parameters()):,}")

    # Test ResNet-32
    model32 = ResNet32()
    y = model32(x)
    print(f"ResNet-32 output shape: {y.shape}")
    print(f"ResNet-32 parameters: {sum(p.numel() for p in model32.parameters()):,}")

    # Test ResNet-50
    model50 = ResNet50()
    y = model50(x)
    print(f"ResNet-50 output shape: {y.shape}")
    print(f"ResNet-50 parameters: {sum(p.numel() for p in model50.parameters()):,}")


ResNet-18 output shape: torch.Size([1, 10])
ResNet-18 parameters: 11,172,810
ResNet-32 output shape: torch.Size([1, 10])
ResNet-32 parameters: 29,984,970
ResNet-50 output shape: torch.Size([1, 10])
ResNet-50 parameters: 23,519,690


In [3]:
"""
Data loading utilities for MNIST and FashionMNIST
Train-Val-Test split: 70%-10%-20%
"""

import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import os


def get_data_loaders(dataset_name='MNIST', batch_size=16, pin_memory=False, num_workers=2):
    """
    Get train, validation, and test data loaders

    Args:
        dataset_name: 'MNIST' or 'FashionMNIST'
        batch_size: Batch size for data loaders
        pin_memory: Whether to pin memory for faster GPU transfer
        num_workers: Number of workers for data loading

    Returns:
        train_loader, val_loader, test_loader
    """

    # Define transforms
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
    ])

    # Data directory
    data_dir = './data'
    os.makedirs(data_dir, exist_ok=True)

    # Load dataset - combine train and test to achieve exact 70-10-20 split
    if dataset_name == 'MNIST':
        train_data = datasets.MNIST(root=data_dir, train=True,
                                    download=True, transform=transform)
        test_data = datasets.MNIST(root=data_dir, train=False,
                                   download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_data = datasets.FashionMNIST(root=data_dir, train=True,
                                           download=True, transform=transform)
        test_data = datasets.FashionMNIST(root=data_dir, train=False,
                                          download=True, transform=transform)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    # Combine both train and test sets for exact 70-10-20 split
    from torch.utils.data import ConcatDataset
    full_dataset = ConcatDataset([train_data, test_data])

    # Split into train (70%), validation (10%), test (20%)
    # Total: 70,000 samples for MNIST/FashionMNIST
    total_size = len(full_dataset)
    train_size = int(0.70 * total_size)  # 70% = 49,000
    val_size = int(0.10 * total_size)     # 10% = 7,000
    test_size = total_size - train_size - val_size  # 20% = 14,000

    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    print(f"\n{dataset_name} Dataset Statistics:")
    print(f"Training samples: {len(train_dataset)} ({len(train_dataset)/(len(train_dataset)+len(val_dataset)+len(test_dataset))*100:.1f}%)")
    print(f"Validation samples: {len(val_dataset)} ({len(val_dataset)/(len(train_dataset)+len(val_dataset)+len(test_dataset))*100:.1f}%)")
    print(f"Test samples: {len(test_dataset)} ({len(test_dataset)/(len(train_dataset)+len(val_dataset)+len(test_dataset))*100:.1f}%)")
    print(f"Total samples: {len(train_dataset) + len(val_dataset) + len(test_dataset)}\n")

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    return train_loader, val_loader, test_loader


def get_svm_data(dataset_name='MNIST'):
    """
    Get flattened data for SVM training

    Args:
        dataset_name: 'MNIST' or 'FashionMNIST'

    Returns:
        X_train, y_train, X_val, y_val, X_test, y_test (as numpy arrays)
    """
    import numpy as np
    from torch.utils.data import ConcatDataset

    # Define simple transform (no normalization for SVM)
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    data_dir = './data'
    os.makedirs(data_dir, exist_ok=True)

    # Load dataset
    if dataset_name == 'MNIST':
        train_data = datasets.MNIST(root=data_dir, train=True,
                                    download=True, transform=transform)
        test_data = datasets.MNIST(root=data_dir, train=False,
                                   download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_data = datasets.FashionMNIST(root=data_dir, train=True,
                                           download=True, transform=transform)
        test_data = datasets.FashionMNIST(root=data_dir, train=False,
                                          download=True, transform=transform)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    # Combine both train and test sets for exact 70-10-20 split
    full_dataset = ConcatDataset([train_data, test_data])

    # Split into train (70%), validation (10%), test (20%)
    total_size = len(full_dataset)
    train_size = int(0.70 * total_size)
    val_size = int(0.10 * total_size)
    test_size = total_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    # Convert to numpy arrays and flatten
    def dataset_to_numpy(dataset):
        X = []
        y = []
        for img, label in dataset:
            X.append(img.numpy().flatten())
            y.append(label)
        return np.array(X), np.array(y)

    X_train, y_train = dataset_to_numpy(train_dataset)
    X_val, y_val = dataset_to_numpy(val_dataset)
    X_test, y_test = dataset_to_numpy(test_dataset)

    print(f"\n{dataset_name} SVM Data Statistics:")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_val shape: {X_val.shape}")
    print(f"X_test shape: {X_test.shape}\n")

    return X_train, y_train, X_val, y_val, X_test, y_test


if __name__ == "__main__":
    # Test data loaders
    print("Testing MNIST data loaders...")
    train_loader, val_loader, test_loader = get_data_loaders('MNIST', batch_size=32)

    print("Testing FashionMNIST data loaders...")
    train_loader, val_loader, test_loader = get_data_loaders('FashionMNIST', batch_size=32)


Testing MNIST data loaders...


100%|██████████| 9.91M/9.91M [00:02<00:00, 4.48MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 138kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.8MB/s]



MNIST Dataset Statistics:
Training samples: 49000 (70.0%)
Validation samples: 7000 (10.0%)
Test samples: 14000 (20.0%)
Total samples: 70000

Testing FashionMNIST data loaders...


100%|██████████| 26.4M/26.4M [00:02<00:00, 8.92MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 188kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.55MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.2MB/s]


FashionMNIST Dataset Statistics:
Training samples: 49000 (70.0%)
Validation samples: 7000 (10.0%)
Test samples: 14000 (20.0%)
Total samples: 70000



In [4]:
"""
Training and evaluation utilities for deep learning models
"""

import torch
import torch.nn as nn
import time
import numpy as np
from tqdm import tqdm


def train_epoch(model, train_loader, criterion, optimizer, device, use_amp=True, scaler=None):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc='Training')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()

        if use_amp and scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        pbar.set_postfix({'loss': running_loss/(pbar.n+1), 'acc': 100.*correct/total})

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc='Validation'):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    val_loss = running_loss / len(val_loader)
    val_acc = 100. * correct / total

    return val_loss, val_acc


def test(model, test_loader, device):
    """Test the model and return accuracy"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc='Testing'):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    test_acc = 100. * correct / total
    return test_acc


def train_model(model, train_loader, val_loader, test_loader,
                epochs, optimizer, criterion, device,
                use_amp=True, model_name='model'):
    """
    Complete training pipeline

    Returns:
        Dictionary with training history and test accuracy
    """
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'train_time': 0,
        'test_acc': 0
    }

    # Initialize GradScaler for mixed precision training
    scaler = torch.cuda.amp.GradScaler() if use_amp and device.type == 'cuda' else None

    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"Device: {device}")
    print(f"Use AMP: {use_amp}")
    print(f"{'='*60}\n")

    start_time = time.time()
    best_val_acc = 0.0

    for epoch in range(epochs):
        print(f"\nEpoch [{epoch+1}/{epochs}]")

        # Train
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, use_amp, scaler
        )

        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'./results/{model_name}_best.pth')

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    train_time = time.time() - start_time
    history['train_time'] = train_time

    # Load best model and test
    model.load_state_dict(torch.load(f'./results/{model_name}_best.pth'))
    test_acc = test(model, test_loader, device)
    history['test_acc'] = test_acc

    print(f"\n{'='*60}")
    print(f"Training completed for {model_name}")
    print(f"Total training time: {train_time:.2f} seconds ({train_time/60:.2f} minutes)")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")
    print(f"Test accuracy: {test_acc:.2f}%")
    print(f"{'='*60}\n")

    return history


def count_flops(model, input_size=(1, 1, 28, 28), device='cpu'):
    """
    Estimate FLOPs for the model
    Using thop library
    """
    try:
        from thop import profile, clever_format
        model = model.to(device)
        input = torch.randn(input_size).to(device)
        flops, params = profile(model, inputs=(input,), verbose=False)
        flops, params = clever_format([flops, params], "%.3f")
        return flops, params
    except ImportError:
        print("Warning: thop not installed. Install with: pip install thop")
        return "N/A", "N/A"


def get_model_info(model, model_name, input_size=(1, 1, 28, 28)):
    """Get model information including parameters and FLOPs"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n{model_name} Information:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    try:
        flops, _ = count_flops(model, input_size)
        print(f"FLOPs: {flops}")
    except:
        print("FLOPs calculation not available")

    return total_params, trainable_params


if __name__ == "__main__":
    print("Training utilities loaded successfully!")


Training utilities loaded successfully!


In [ ]:
# Import custom modules (adjust paths if needed)
# If running locally, ensure the modules are in the correct path
# If in Colab, upload the files or clone from GitHub

# For Colab, you might need to:
# !git clone https://github.com/YourUsername/MLOps-YourName-YourRollNumber.git
# %cd MLOps-YourName-YourRollNumber

from models.resnet import ResNet18, ResNet32, ResNet50
from utils.data_loader import get_data_loaders, get_svm_data
from utils.train import train_model, count_flops, get_model_info, test
from models.svm_classifier import train_svm_variants, print_svm_results

ModuleNotFoundError: No module named 'models'

## 2. Configuration and Hyperparameters

In [15]:
# Global configuration
USE_AMP = True  # Automatic Mixed Precision
NUM_WORKERS = 2

# Experiment configurations for Q1(a) with pin_memory=True
q2a_configs = [
    # Batch size 16 with pinned memory
    #{'batch_size': 16, 'optimizer': 'SGD', 'lr': 0.001, 'epochs': 5, 'pin_memory': True},
    #{'batch_size': 16, 'optimizer': 'SGD', 'lr': 0.0001, 'epochs': 5, 'pin_memory': True},
    #{'batch_size': 16, 'optimizer': 'Adam', 'lr': 0.001, 'epochs': 5, 'pin_memory': True},
    #{'batch_size': 16, 'optimizer': 'Adam', 'lr': 0.0001, 'epochs': 5, 'pin_memory': True},

    # Batch size 32 with pinned memory
    {'batch_size': 32, 'optimizer': 'SGD', 'lr': 0.001, 'epochs': 5, 'pin_memory': True},
    {'batch_size': 32, 'optimizer': 'SGD', 'lr': 0.0001, 'epochs': 5, 'pin_memory': True},
    {'batch_size': 32, 'optimizer': 'Adam', 'lr': 0.001, 'epochs': 5, 'pin_memory': True},
    {'batch_size': 32, 'optimizer': 'Adam', 'lr': 0.0001, 'epochs': 5, 'pin_memory': True},
]

print(f"Total Q1(a) experiments per model per dataset: {len(q2a_configs)}")
print(f"All configurations use pin_memory=True for faster GPU data transfer")

Total Q1(a) experiments per model per dataset: 2
All configurations use pin_memory=True for faster GPU data transfer


## 3. Q1(a): Deep Learning Models on MNIST and FashionMNIST

### 3.1 MNIST Experiments

In [9]:
# Dictionary to store all results
mnist_results = {
    'ResNet-18': [],
    'ResNet-50': []
}

dataset_name = 'MNIST'

In [16]:
# Train ResNet-18 on MNIST with all configurations
print("="*80)
print("TRAINING RESNET-18 ON MNIST")
print("="*80)

# Create results directory if it doesn't exist
import os
os.makedirs('./results', exist_ok=True)

for config in q2a_configs:
    print(f"\n{'='*60}")
    print(f"Config: {config}")
    print(f"{'='*60}\n")

    # Get data loaders with pinned memory enabled
    pin_mem = config.get('pin_memory', True)
    train_loader, val_loader, test_loader = get_data_loaders(
        dataset_name=dataset_name,
        batch_size=config['batch_size'],
        pin_memory=pin_mem,
        num_workers=NUM_WORKERS
    )

    # Initialize model
    model = ResNet18(num_classes=10, in_channels=1, pretrained=False)
    model = model.to(device)

    # Setup optimizer
    if config['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9, weight_decay=5e-4)
    else:  # Adam
        optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=5e-4)

    criterion = nn.CrossEntropyLoss()

    # Train model
    model_name = f"mnist_resnet18_bs{config['batch_size']}_{config['optimizer']}_lr{config['lr']}"
    history = train_model(
        model, train_loader, val_loader, test_loader,
        epochs=config['epochs'],
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        use_amp=USE_AMP,
        model_name=model_name
    )

    # Store results
    result = {
        'config': config,
        'history': history,
        'test_accuracy': history['test_acc'],
        'train_time': history['train_time']
    }
    mnist_results['ResNet-18'].append(result)

    print(f"\nCompleted: {model_name}")
    print(f"Test Accuracy: {history['test_acc']:.2f}%")
    print(f"Training Time: {history['train_time']:.2f}s\n")

TRAINING RESNET-18 ON MNIST

Config: {'batch_size': 32, 'optimizer': 'Adam', 'lr': 0.001, 'epochs': 5, 'pin_memory': True}


MNIST Dataset Statistics:
Training samples: 49000 (70.0%)
Validation samples: 7000 (10.0%)
Test samples: 14000 (20.0%)
Total samples: 70000


Training mnist_resnet18_bs32_Adam_lr0.001
Device: cuda
Use AMP: True


Epoch [1/5]


Validation: 100%|██████████| 219/219 [00:02<00:00, 93.25it/s]


Train Loss: 0.1461, Train Acc: 95.57%
Val Loss: 0.0808, Val Acc: 97.70%

Epoch [2/5]


Validation: 100%|██████████| 219/219 [00:02<00:00, 94.98it/s] 


Train Loss: 0.0952, Train Acc: 97.26%
Val Loss: 0.0639, Val Acc: 97.97%

Epoch [3/5]


...

Train Loss: 0.0322, Train Acc: 99.05%
Val Loss: 0.0257, Val Acc: 99.34%

Epoch [5/5]


Validation: 100%|██████████| 219/219 [00:02<00:00, 79.27it/s] 


Train Loss: 0.0299, Train Acc: 99.14%
Val Loss: 0.0382, Val Acc: 98.80%


Testing: 100%|██████████| 438/438 [00:04<00:00, 98.45it/s]


Training completed for mnist_resnet18_bs32_Adam_lr0.0001
Total training time: 208.33 seconds (3.47 minutes)
Best validation accuracy: 99.34%
Test accuracy: 99.15%


Completed: mnist_resnet18_bs32_Adam_lr0.0001
Test Accuracy: 99.15%
Training Time: 208.33s



In [ ]:
# Train ResNet-50 on MNIST with all configurations
print("="*80)
print("TRAINING RESNET-50 ON MNIST")
print("="*80)

for config in q2a_configs:
    print(f"\n{'='*60}")
    print(f"Config: {config}")
    print(f"{'='*60}\n")

    # Get data loaders with pinned memory enabled
    pin_mem = config.get('pin_memory', True)
    train_loader, val_loader, test_loader = get_data_loaders(
        dataset_name=dataset_name,
        batch_size=config['batch_size'],
        pin_memory=pin_mem,
        num_workers=NUM_WORKERS
    )

    # Initialize model
    model = ResNet50(num_classes=10, in_channels=1, pretrained=False)
    model = model.to(device)

    # Setup optimizer
    if config['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9, weight_decay=5e-4)
    else:  # Adam
        optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=5e-4)

    criterion = nn.CrossEntropyLoss()

    # Train model
    model_name = f"mnist_resnet50_bs{config['batch_size']}_{config['optimizer']}_lr{config['lr']}"
    history = train_model(
        model, train_loader, val_loader, test_loader,
        epochs=config['epochs'],
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        use_amp=USE_AMP,
        model_name=model_name
    )

    # Store results
    result = {
        'config': config,
        'history': history,
        'test_accuracy': history['test_acc'],
        'train_time': history['train_time']
    }
    mnist_results['ResNet-50'].append(result)

    print(f"\nCompleted: {model_name}")
    print(f"Test Accuracy: {history['test_acc']:.2f}%")
    print(f"Training Time: {history['train_time']:.2f}s\n")

TRAINING RESNET-50 ON MNIST

Config: {'batch_size': 32, 'optimizer': 'Adam', 'lr': 0.001, 'epochs': 5, 'pin_memory': True}


MNIST Dataset Statistics:
Training samples: 49000 (70.0%)
Validation samples: 7000 (10.0%)
Test samples: 14000 (20.0%)
Total samples: 70000


Training mnist_resnet50_bs32_Adam_lr0.001
Device: cuda
Use AMP: True


Epoch [1/5]


Validation: 100%|██████████| 219/219 [00:05<00:00, 38.08it/s]


Train Loss: 0.2161, Train Acc: 93.55%
Val Loss: 0.1533, Val Acc: 95.74%

Epoch [2/5]


Validation: 100%|██████████| 219/219 [00:05<00:00, 37.88it/s]


Train Loss: 0.1253, Train Acc: 96.38%
Val Loss: 0.1601, Val Acc: 95.26%

Epoch [3/5]


Training:  66%|██████▌   | 1009/1532 [00:54<00:28, 18.67it/s, loss=0.106, acc=96.8]

### 3.2 FashionMNIST Experiments

In [ ]:
# Dictionary to store all results
fashion_mnist_results = {
    'ResNet-18': [],
    'ResNet-50': []
}

dataset_name = 'FashionMNIST'

In [ ]:
# Train ResNet-18 on FashionMNIST
print("="*80)
print("TRAINING RESNET-18 ON FASHIONMNIST")
print("="*80)

# Create results directory if it doesn't exist
import os
os.makedirs('./results', exist_ok=True)

for config in q2a_configs:
    print(f"\n{'='*60}")
    print(f"Config: {config}")
    print(f"{'='*60}\n")

    # Get data loaders with pinned memory enabled
    pin_mem = config.get('pin_memory', True)
    train_loader, val_loader, test_loader = get_data_loaders(
        dataset_name=dataset_name,
        batch_size=config['batch_size'],
        pin_memory=pin_mem,
        num_workers=NUM_WORKERS
    )

    # Initialize model
    model = ResNet18(num_classes=10, in_channels=1, pretrained=False)
    model = model.to(device)

    # Setup optimizer
    if config['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9, weight_decay=5e-4)
    else:  # Adam
        optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=5e-4)

    criterion = nn.CrossEntropyLoss()

    # Train model
    model_name = f"fashion_resnet18_bs{config['batch_size']}_{config['optimizer']}_lr{config['lr']}_pinned"
    history = train_model(
        model, train_loader, val_loader, test_loader,
        epochs=config['epochs'],
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        use_amp=USE_AMP,
        model_name=model_name
    )

    # Store results
    result = {
        'config': config,
        'history': history,
        'test_accuracy': history['test_acc'],
        'train_time': history['train_time']
    }
    fashion_mnist_results['ResNet-18'].append(result)

    print(f"\nCompleted: {model_name}")
    print(f"Test Accuracy: {history['test_acc']:.2f}%")
    print(f"Training Time: {history['train_time']:.2f}s\n")

TRAINING RESNET-18 ON FASHIONMNIST

Config: {'batch_size': 16, 'optimizer': 'Adam', 'lr': 0.001, 'epochs': 10}


FashionMNIST Dataset Statistics:
Training samples: 52500 (75.0%)
Validation samples: 7500 (10.7%)
Test samples: 10000 (14.3%)
Total samples: 70000


Training fashion_resnet18_bs16_Adam_lr0.001
Device: cuda
Use AMP: True


Epoch [1/10]


Validation: 100%|██████████| 469/469 [00:04<00:00, 115.96it/s]


Train Loss: 0.4807, Train Acc: 82.91%
Val Loss: 0.3281, Val Acc: 88.19%

Epoch [2/10]


Validation: 100%|██████████| 469/469 [00:03<00:00, 128.23it/s]


Train Loss: 0.3391, Train Acc: 88.02%
Val Loss: 0.3357, Val Acc: 88.51%

Epoch [3/10]


...

Train Loss: 0.2109, Train Acc: 92.54%
Val Loss: 0.2301, Val Acc: 91.48%

Epoch [10/10]


Validation: 100%|██████████| 469/469 [00:03<00:00, 131.08it/s]


Train Loss: 0.2042, Train Acc: 92.73%
Val Loss: 0.2324, Val Acc: 91.65%


Testing: 100%|██████████| 625/625 [00:05<00:00, 116.33it/s]


Training completed for fashion_resnet18_bs16_Adam_lr0.001
Total training time: 749.31 seconds (12.49 minutes)
Best validation accuracy: 92.11%
Test accuracy: 91.42%


Completed: fashion_resnet18_bs16_Adam_lr0.001
Test Accuracy: 91.42%
Training Time: 749.31s



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# Train ResNet-50 on FashionMNIST
print("="*80)
print("TRAINING RESNET-50 ON FASHIONMNIST")
print("="*80)

for config in q2a_configs:
    print(f"\n{'='*60}")
    print(f"Config: {config}")
    print(f"{'='*60}\n")

    # Get data loaders with pinned memory enabled
    pin_mem = config.get('pin_memory', True)
    train_loader, val_loader, test_loader = get_data_loaders(
        dataset_name=dataset_name,
        batch_size=config['batch_size'],
        pin_memory=pin_mem,
        num_workers=NUM_WORKERS
    )

    # Initialize model
    model = ResNet50(num_classes=10, in_channels=1, pretrained=False)
    model = model.to(device)

    # Setup optimizer
    if config['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9, weight_decay=5e-4)
    else:  # Adam
        optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=5e-4)

    criterion = nn.CrossEntropyLoss()

    # Train model
    model_name = f"fashion_resnet50_bs{config['batch_size']}_{config['optimizer']}_lr{config['lr']}_pinned"
    history = train_model(
        model, train_loader, val_loader, test_loader,
        epochs=config['epochs'],
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        use_amp=USE_AMP,
        model_name=model_name
    )

    # Store results
    result = {
        'config': config,
        'history': history,
        'test_accuracy': history['test_acc'],
        'train_time': history['train_time']
    }
    fashion_mnist_results['ResNet-50'].append(result)

    print(f"\nCompleted: {model_name}")
    print(f"Test Accuracy: {history['test_acc']:.2f}%")
    print(f"Training Time: {history['train_time']:.2f}s\n")

TRAINING RESNET-50 ON FASHIONMNIST

Config: {'batch_size': 16, 'optimizer': 'Adam', 'lr': 0.001, 'epochs': 10}


FashionMNIST Dataset Statistics:
Training samples: 52500 (75.0%)
Validation samples: 7500 (10.7%)
Test samples: 10000 (14.3%)
Total samples: 70000


Training fashion_resnet50_bs16_Adam_lr0.001
Device: cuda
Use AMP: True


Epoch [1/10]


Validation: 100%|██████████| 469/469 [00:08<00:00, 54.16it/s]


Train Loss: 0.5697, Train Acc: 79.74%
Val Loss: 0.3969, Val Acc: 85.77%

Epoch [2/10]


Validation: 100%|██████████| 469/469 [00:07<00:00, 61.63it/s]


Train Loss: 0.3819, Train Acc: 86.37%
Val Loss: 0.3303, Val Acc: 87.87%

Epoch [3/10]


...

Train Loss: 0.2426, Train Acc: 91.50%
Val Loss: 0.2642, Val Acc: 90.37%

Epoch [10/10]


Validation: 100%|██████████| 469/469 [00:07<00:00, 63.97it/s]


Train Loss: 0.2383, Train Acc: 91.50%
Val Loss: 0.2396, Val Acc: 91.60%


Testing: 100%|██████████| 625/625 [00:09<00:00, 64.18it/s]


Training completed for fashion_resnet50_bs16_Adam_lr0.001
Total training time: 1473.59 seconds (24.56 minutes)
Best validation accuracy: 91.60%
Test accuracy: 91.12%


Completed: fashion_resnet50_bs16_Adam_lr0.001
Test Accuracy: 91.12%
Training Time: 1473.59s



### 3.3 Q1(a) Results Summary

In [ ]:
# Create results table for MNIST
def create_results_table(results_dict, dataset_name):
    data = []
    for model_name, results in results_dict.items():
        for result in results:
            config = result['config']
            data.append({
                'Dataset': dataset_name,
                'Model': model_name,
                'Batch Size': config['batch_size'],
                'Optimizer': config['optimizer'],
                'Learning Rate': config['lr'],
                'Epochs': config['epochs'],
                'Test Accuracy (%)': f"{result['test_accuracy']:.2f}",
                'Train Time (s)': f"{result['train_time']:.2f}"
            })
    return pd.DataFrame(data)

# MNIST Results
print("\n" + "="*80)
print("MNIST RESULTS")
print("="*80)
mnist_df = create_results_table(mnist_results, 'MNIST')
print(mnist_df.to_string(index=False))

# FashionMNIST Results
print("\n" + "="*80)
print("FASHIONMNIST RESULTS")
print("="*80)
fashion_df = create_results_table(fashion_mnist_results, 'FashionMNIST')
print(fashion_df.to_string(index=False))

## 4. Q1(b): SVM Classifier

In [ ]:
# Load data for SVM (flattened images)
print("Loading MNIST data for SVM...")
X_train_mnist, y_train_mnist, X_val_mnist, y_val_mnist, X_test_mnist, y_test_mnist = get_svm_data('MNIST')

print("\nLoading FashionMNIST data for SVM...")
X_train_fashion, y_train_fashion, X_val_fashion, y_val_fashion, X_test_fashion, y_test_fashion = get_svm_data('FashionMNIST')

In [ ]:
# Train SVM on MNIST
print("\n" + "="*80)
print("TRAINING SVM ON MNIST")
print("="*80)
mnist_svm_results = train_svm_variants(
    X_train_mnist, y_train_mnist,
    X_val_mnist, y_val_mnist,
    X_test_mnist, y_test_mnist,
    dataset_name='MNIST'
)

In [ ]:
# Train SVM on FashionMNIST
print("\n" + "="*80)
print("TRAINING SVM ON FASHIONMNIST")
print("="*80)
fashion_svm_results = train_svm_variants(
    X_train_fashion, y_train_fashion,
    X_val_fashion, y_val_fashion,
    X_test_fashion, y_test_fashion,
    dataset_name='FashionMNIST'
)

In [ ]:
# Display SVM results
print("\n" + "="*80)
print("SVM RESULTS - MNIST")
print("="*80)
print_svm_results(mnist_svm_results)

print("\n" + "="*80)
print("SVM RESULTS - FASHIONMNIST")
print("="*80)
print_svm_results(fashion_svm_results)

## 5. Q2: CPU vs GPU Performance Comparison

In [ ]:
# Q2 Configuration
q2_configs = [
    {'batch_size': 16, 'optimizer': 'SGD', 'lr': 0.001, 'epochs': 5},
    {'batch_size': 16, 'optimizer': 'Adam', 'lr': 0.001, 'epochs': 5},
]

models_q2 = ['ResNet-18', 'ResNet-32', 'ResNet-50']
devices_q2 = ['cpu', 'cuda'] if torch.cuda.is_available() else ['cpu']

q2_results = []

In [ ]:
# Calculate FLOPs for each model
print("Calculating FLOPs for models...\n")
flops_info = {}

for model_name in models_q2:
    if model_name == 'ResNet-18':
        model = ResNet18()
    elif model_name == 'ResNet-32':
        model = ResNet32()
    else:
        model = ResNet50()

    flops, params = count_flops(model, input_size=(1, 1, 28, 28), device='cpu')
    flops_info[model_name] = {'flops': flops, 'params': params}
    print(f"{model_name}: FLOPs = {flops}, Parameters = {params}")

In [ ]:
# Run Q2 experiments
print("\n" + "="*80)
print("Q2: CPU vs GPU COMPARISON ON FASHIONMNIST")
print("="*80)

for device_name in devices_q2:
    dev = torch.device(device_name)
    print(f"\n{'='*60}")
    print(f"Running experiments on: {device_name.upper()}")
    print(f"{'='*60}\n")

    for config in q2_configs:
        # Get data loaders
        train_loader, val_loader, test_loader = get_data_loaders(
            dataset_name='FashionMNIST',
            batch_size=config['batch_size'],
            pin_memory=(device_name == 'cuda'),
            num_workers=NUM_WORKERS
        )

        for model_name in models_q2:
            print(f"\n--- {model_name} on {device_name.upper()} ---")
            print(f"Config: {config}\n")

            # Initialize model
            if model_name == 'ResNet-18':
                model = ResNet18(num_classes=10, in_channels=1, pretrained=False)
            elif model_name == 'ResNet-32':
                model = ResNet32(num_classes=10, in_channels=1, pretrained=False)
            else:
                model = ResNet50(num_classes=10, in_channels=1, pretrained=False)

            model = model.to(dev)

            # Setup optimizer
            if config['optimizer'] == 'SGD':
                optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9)
            else:
                optimizer = optim.Adam(model.parameters(), lr=config['lr'])

            criterion = nn.CrossEntropyLoss()

            # Train model
            save_name = f"q2_fashion_{model_name}_{device_name}_bs{config['batch_size']}_{config['optimizer']}"
            use_amp_q2 = USE_AMP if device_name == 'cuda' else False

            history = train_model(
                model, train_loader, val_loader, test_loader,
                epochs=config['epochs'],
                optimizer=optimizer,
                criterion=criterion,
                device=dev,
                use_amp=use_amp_q2,
                model_name=save_name
            )

            # Store results
            result = {
                'device': device_name.upper(),
                'model': model_name,
                'batch_size': config['batch_size'],
                'optimizer': config['optimizer'],
                'lr': config['lr'],
                'test_accuracy': history['test_acc'],
                'train_time_ms': history['train_time'] * 1000,  # Convert to ms
                'flops': flops_info[model_name]['flops'],
                'params': flops_info[model_name]['params']
            }
            q2_results.append(result)

            print(f"\nCompleted: {save_name}")
            print(f"Test Accuracy: {history['test_acc']:.2f}%")
            print(f"Training Time: {history['train_time']*1000:.2f}ms\n")

In [ ]:
# Q2 Results Summary
q2_df = pd.DataFrame(q2_results)
print("\n" + "="*80)
print("Q2 RESULTS SUMMARY")
print("="*80)
print(q2_df.to_string(index=False))

# Calculate speedup (GPU vs CPU)
if 'cuda' in devices_q2:
    print("\n" + "="*80)
    print("GPU SPEEDUP ANALYSIS")
    print("="*80)
    for model_name in models_q2:
        cpu_time = q2_df[(q2_df['device'] == 'CPU') & (q2_df['model'] == model_name)]['train_time_ms'].mean()
        gpu_time = q2_df[(q2_df['device'] == 'CUDA') & (q2_df['model'] == model_name)]['train_time_ms'].mean()
        speedup = cpu_time / gpu_time if gpu_time > 0 else 0
        print(f"{model_name}: {speedup:.2f}x speedup (CPU: {cpu_time:.2f}ms, GPU: {gpu_time:.2f}ms)")

## 6. Visualization

In [ ]:
# Plot training curves for a sample experiment
def plot_training_curves(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    epochs = range(1, len(history['train_loss']) + 1)
    ax1.plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'{title} - Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Accuracy curves
    ax2.plot(epochs, history['train_acc'], 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title(f'{title} - Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'./results/{title.replace(" ", "_")}_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot best performing models
print("\nPlotting training curves for best models...\n")

# Find best MNIST ResNet-18 result
best_mnist_resnet18 = max(mnist_results['ResNet-18'], key=lambda x: x['test_accuracy'])
plot_training_curves(best_mnist_resnet18['history'], 'MNIST ResNet-18 (Best)')

# Find best FashionMNIST ResNet-18 result
best_fashion_resnet18 = max(fashion_mnist_results['ResNet-18'], key=lambda x: x['test_accuracy'])
plot_training_curves(best_fashion_resnet18['history'], 'FashionMNIST ResNet-18 (Best)')

In [ ]:
# Comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy comparison across batch sizes
ax = axes[0, 0]
for model_name in ['ResNet-18', 'ResNet-50']:
    accs = [r['test_accuracy'] for r in fashion_mnist_results[model_name][:4]]  # First 4 configs (bs=16)
    ax.plot(range(1, 5), accs, marker='o', label=model_name, linewidth=2, markersize=8)
ax.set_xlabel('Configuration', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('FashionMNIST: Batch Size 16 Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Training time comparison
ax = axes[0, 1]
models = ['ResNet-18', 'ResNet-50']
times_mnist = [np.mean([r['train_time'] for r in mnist_results[m]]) for m in models]
times_fashion = [np.mean([r['train_time'] for r in fashion_mnist_results[m]]) for m in models]
x = np.arange(len(models))
width = 0.35
ax.bar(x - width/2, times_mnist, width, label='MNIST', alpha=0.8)
ax.bar(x + width/2, times_fashion, width, label='FashionMNIST', alpha=0.8)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Average Training Time (s)', fontsize=12)
ax.set_title('Average Training Time Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Optimizer comparison
ax = axes[1, 0]
sgd_accs = [r['test_accuracy'] for r in fashion_mnist_results['ResNet-18'] if r['config']['optimizer'] == 'SGD']
adam_accs = [r['test_accuracy'] for r in fashion_mnist_results['ResNet-18'] if r['config']['optimizer'] == 'Adam']
ax.boxplot([sgd_accs, adam_accs], labels=['SGD', 'Adam'])
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('FashionMNIST ResNet-18: Optimizer Comparison', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: CPU vs GPU (if available)
ax = axes[1, 1]
if len(q2_results) > 0:
    cpu_results = [r for r in q2_results if r['device'] == 'CPU']
    gpu_results = [r for r in q2_results if r['device'] == 'CUDA']

    if cpu_results and gpu_results:
        cpu_times = [r['train_time_ms'] for r in cpu_results]
        gpu_times = [r['train_time_ms'] for r in gpu_results]
        model_labels = [r['model'] for r in cpu_results]

        x = np.arange(len(model_labels))
        width = 0.35
        ax.bar(x - width/2, cpu_times, width, label='CPU', alpha=0.8)
        ax.bar(x + width/2, gpu_times, width, label='GPU', alpha=0.8)
        ax.set_xlabel('Model', fontsize=12)
        ax.set_ylabel('Training Time (ms)', fontsize=12)
        ax.set_title('CPU vs GPU Training Time', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(model_labels, rotation=45, ha='right')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('./results/comparison_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Confusion Matrix for Best Models

In [ ]:
"""
Generate Confusion Matrices for Best Models
Best MNIST Model: ResNet-18, BS=16, SGD, LR=0.001 (99.81% accuracy)
Best FashionMNIST Model: ResNet-18, BS=16, Adam, LR=0.0001 (97.47% accuracy)
"""

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import os

# Import model and data loader
from models.resnet import ResNet18
from utils.data_loader import get_data_loaders

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Class labels
MNIST_CLASSES = [str(i) for i in range(10)]
FASHION_CLASSES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


def get_predictions(model, data_loader, device):
    """Get all predictions and true labels from a data loader"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(data_loader, desc='Getting predictions'):
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(targets.numpy())
    
    return np.array(all_preds), np.array(all_labels)


def plot_confusion_matrix(cm, class_names, title, save_path=None):
    """Plot a confusion matrix with annotations"""
    plt.figure(figsize=(12, 10))
    
    # Normalize confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Create heatmap
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                square=True, cbar_kws={'shrink': 0.8})
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Confusion matrix saved to: {save_path}")
    
    plt.show()
    
    return cm_normalized


def generate_confusion_matrix_for_model(dataset_name, model_path, class_names, results_dir='./results'):
    """Generate and plot confusion matrix for a specific model"""
    
    print(f"\n{'='*60}")
    print(f"Generating Confusion Matrix for {dataset_name}")
    print(f"Model path: {model_path}")
    print(f"{'='*60}")
    
    # Load data
    _, _, test_loader = get_data_loaders(dataset_name, batch_size=64, pin_memory=False)
    
    # Create model
    model = ResNet18(num_classes=10).to(device)
    
    # Load trained weights
    full_model_path = os.path.join(results_dir, model_path)
    if not os.path.exists(full_model_path):
        print(f"Error: Model file not found at {full_model_path}")
        return None
    
    model.load_state_dict(torch.load(full_model_path, map_location=device))
    print(f"Loaded model weights from: {full_model_path}")
    
    # Get predictions
    predictions, true_labels = get_predictions(model, test_loader, device)
    
    # Calculate confusion matrix
    cm = confusion_matrix(true_labels, predictions)
    
    # Plot confusion matrix
    title = f'Confusion Matrix - Best {dataset_name} Model\n(ResNet-18, Test Set)'
    save_path = os.path.join(results_dir, f'{dataset_name.lower()}_confusion_matrix.png')
    cm_normalized = plot_confusion_matrix(cm, class_names, title, save_path)
    
    # Print classification report
    print(f"\nClassification Report for {dataset_name}:")
    print(classification_report(true_labels, predictions, target_names=class_names))
    
    # Calculate per-class accuracy
    print(f"\nPer-Class Accuracy for {dataset_name}:")
    for i, class_name in enumerate(class_names):
        class_acc = cm[i, i] / cm[i].sum() * 100
        print(f"  {class_name}: {class_acc:.2f}%")
    
    return cm, cm_normalized


# Results directory where trained models are saved
RESULTS_DIR = '../results_with_pinmemory=false_epochs=10'

# Generate confusion matrix for best MNIST model
print("\n" + "="*70)
print("CONFUSION MATRIX ANALYSIS FOR BEST MODELS")
print("="*70)

# Best MNIST model: ResNet-18, BS=16, SGD, LR=0.001
mnist_model_path = 'mnist_resnet18_bs16_SGD_lr0.001_best.pth'
mnist_cm, mnist_cm_norm = generate_confusion_matrix_for_model(
    'MNIST', mnist_model_path, MNIST_CLASSES, RESULTS_DIR
)

# Best FashionMNIST model: ResNet-18, BS=16, Adam, LR=0.0001
fashion_model_path = 'fashion_resnet18_bs16_Adam_lr0.0001_best.pth'
fashion_cm, fashion_cm_norm = generate_confusion_matrix_for_model(
    'FashionMNIST', fashion_model_path, FASHION_CLASSES, RESULTS_DIR
)

In [ ]:
# Side-by-side confusion matrix comparison
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# MNIST confusion matrix
if mnist_cm_norm is not None:
    sns.heatmap(mnist_cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=MNIST_CLASSES, yticklabels=MNIST_CLASSES,
                square=True, ax=axes[0], cbar_kws={'shrink': 0.8})
    axes[0].set_title('MNIST - Best Model (ResNet-18)\nBS=16, SGD, LR=0.001', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Predicted Label', fontsize=10)
    axes[0].set_ylabel('True Label', fontsize=10)

# FashionMNIST confusion matrix
if fashion_cm_norm is not None:
    sns.heatmap(fashion_cm_norm, annot=True, fmt='.2%', cmap='Oranges',
                xticklabels=FASHION_CLASSES, yticklabels=FASHION_CLASSES,
                square=True, ax=axes[1], cbar_kws={'shrink': 0.8})
    axes[1].set_title('FashionMNIST - Best Model (ResNet-18)\nBS=16, Adam, LR=0.0001', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Predicted Label', fontsize=10)
    axes[1].set_ylabel('True Label', fontsize=10)
    axes[1].set_xticklabels(FASHION_CLASSES, rotation=45, ha='right')

plt.suptitle('Confusion Matrices Comparison - Best Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrices_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("ANALYSIS SUMMARY")
print("="*70)
print("\nBest MNIST Model: ResNet-18, Batch Size=16, SGD, LR=0.001")
print(f"  Overall Test Accuracy: 99.81%")
print("\nBest FashionMNIST Model: ResNet-18, Batch Size=16, Adam, LR=0.0001")
print(f"  Overall Test Accuracy: 97.47%")
print("\nConfusion matrices have been saved to the results directory.")

## 7. Analysis and Conclusions

### Key Findings

Write your detailed analysis here based on the results:

#### Q1(a) Analysis
- **Batch Size Impact:**
- **Optimizer Comparison:**
- **Learning Rate Effects:**
- **Model Comparison (ResNet-18 vs ResNet-50):**

#### Q1(b) Analysis
- **SVM Performance:**
- **Kernel Comparison:**
- **SVM vs Deep Learning:**

#### Q2 Analysis
- **CPU vs GPU:**
- **Model Complexity:**
- **FLOPs and Efficiency:**

### Recommendations
1.
2.
3.

### Limitations
1.
2.
3.

## 8. Save Results

In [ ]:
# Save all results to CSV files
import json

# Save Q1a results
mnist_df.to_csv('./results/mnist_q1a_results.csv', index=False)
fashion_df.to_csv('./results/fashionmnist_q1a_results.csv', index=False)

# Save Q1b SVM results
pd.DataFrame(mnist_svm_results).to_csv('./results/mnist_svm_results.csv', index=False)
pd.DataFrame(fashion_svm_results).to_csv('./results/fashionmnist_svm_results.csv', index=False)

# Save Q2 results
q2_df.to_csv('./results/q2_cpu_gpu_results.csv', index=False)

# Save complete results as JSON for later analysis
all_results = {
    'mnist_dl': mnist_results,
    'fashion_mnist_dl': fashion_mnist_results,
    'mnist_svm': mnist_svm_results,
    'fashion_mnist_svm': fashion_svm_results,
    'q2_cpu_gpu': q2_results,
    'flops_info': flops_info
}

with open('./results/all_results.json', 'w') as f:
    # Convert numpy types to native Python types for JSON serialization
    json.dump(all_results, f, indent=2, default=str)

print("All results saved successfully!")
print("\nSaved files:")
print("- ./results/mnist_q1a_results.csv")
print("- ./results/fashionmnist_q1a_results.csv")
print("- ./results/mnist_svm_results.csv")
print("- ./results/fashionmnist_svm_results.csv")
print("- ./results/q2_cpu_gpu_results.csv")
print("- ./results/all_results.json")

## End of Assignment

**Remember to:**
1. Update your README.md with the results
2. Share this Colab notebook link in your report
3. Create a comprehensive report PDF with all results and analysis
4. Push all code, results, and report to GitHub
5. Create GitHub pages for your repository

## 9. Extract Test Accuracies from Saved Models

Run these cells to load all saved .pth files and evaluate them to get test accuracies.

In [ ]:
# Define evaluation function
def parse_filename(filename):
    """Parse model filename to extract configuration"""
    pattern = r'(\w+)_resnet(\d+)_bs(\d+)_(\w+)_lr([\d.]+)_best\.pth'
    match = re.match(pattern, filename)

    if match:
        dataset, version, bs, opt, lr = match.groups()
        return {
            'Dataset': dataset.upper(),
            'Model': f'ResNet-{version}',
            'Batch Size': int(bs),
            'Optimizer': opt,
            'Learning Rate': float(lr)
        }
    return None

def evaluate_saved_model(model_path, config, device='cuda'):
    """Load model and evaluate on test set"""
    dataset_name = 'MNIST' if config['Dataset'] == 'MNIST' else 'FashionMNIST'

    print(f"Evaluating: {config['Dataset']} {config['Model']} BS={config['Batch Size']} {config['Optimizer']} LR={config['Learning Rate']}")

    # Load data
    _, _, test_loader = get_data_loaders(
        dataset_name=dataset_name,
        batch_size=config['Batch Size'],
        pin_memory=False,
        num_workers=0
    )

    # Initialize model
    if config['Model'] == 'ResNet-18':
        model = ResNet18(num_classes=10, in_channels=1, pretrained=False)
    else:  # ResNet-50
        model = ResNet50(num_classes=10, in_channels=1, pretrained=False)

    # Load weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)

    # Evaluate
    test_acc = test(model, test_loader, device)
    print(f"  ✓ Test Accuracy: {test_acc:.2f}%\n")

    return test_acc

print("Evaluation functions defined successfully!")

In [ ]:
# Evaluate all saved models
print("="*80)
print("EVALUATING ALL SAVED MODELS")
print("="*80)

results_dir = Path('./results')
pth_files = sorted(results_dir.glob('*.pth'))

print(f"\nFound {len(pth_files)} saved models\n")

all_results = []

for pth_file in pth_files:
    config = parse_filename(pth_file.name)

    if config is None:
        print(f"Skipping {pth_file.name} - could not parse filename\n")
        continue

    try:
        test_acc = evaluate_saved_model(str(pth_file), config, device)
        config['Test Accuracy (%)'] = round(test_acc, 2)
        all_results.append(config)

    except Exception as e:
        print(f"✗ Error evaluating {pth_file.name}: {e}\n")

print("\n" + "="*80)
print(f"EVALUATION COMPLETE - {len(all_results)} models evaluated successfully")
print("="*80)

In [ ]:
# Create result tables
results_df = pd.DataFrame(all_results)

# Save complete results
results_df.to_csv('./results/all_models_with_accuracies.csv', index=False)
print("\n✓ Saved: all_models_with_accuracies.csv")

# Process each dataset
for dataset in ['MNIST', 'FASHION']:
    dataset_df = results_df[results_df['Dataset'] == dataset].copy()

    if len(dataset_df) == 0:
        continue

    # Create pivot table in assignment format
    pivot = dataset_df.pivot_table(
        index=['Batch Size', 'Optimizer', 'Learning Rate'],
        columns='Model',
        values='Test Accuracy (%)',
        aggfunc='first'
    ).reset_index()

    # Reorder columns
    cols = ['Batch Size', 'Optimizer', 'Learning Rate']
    model_cols = [col for col in pivot.columns if col.startswith('ResNet')]
    pivot = pivot[cols + model_cols]

    # Save
    filename = f'./results/{dataset.lower()}_complete_results.csv'
    pivot.to_csv(filename, index=False)
    print(f"✓ Saved: {filename}")

    # Display
    print(f"\n{'='*80}")
    print(f"{dataset} - COMPLETE RESULTS")
    print(f"{'='*80}")
    print(pivot.to_string(index=False))

    # Find best configuration
    best_idx = dataset_df['Test Accuracy (%)'].idxmax()
    best = dataset_df.loc[best_idx]
    print(f"\n🏆 Best {dataset} Configuration:")
    print(f"   Accuracy: {best['Test Accuracy (%)']}%")
    print(f"   Model: {best['Model']}")
    print(f"   Batch Size: {best['Batch Size']}")
    print(f"   Optimizer: {best['Optimizer']}")
    print(f"   Learning Rate: {best['Learning Rate']}")

print("\n" + "="*80)
print("✅ ALL RESULTS EXTRACTED AND SAVED!")
print("="*80)
print("\nGenerated files:")
print("  - all_models_with_accuracies.csv")
print("  - mnist_complete_results.csv")
print("  - fashion_complete_results.csv")